# 01 — Data Import & Cleaning

Load all four EmoSurv CSVs and produce a single cleaned dataset.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

## Load raw datasets

In [ ]:
participants = pd.read_csv('../data/raw/participants.csv', sep=';')
fixed = pd.read_csv('../data/raw/fixed_text.csv', sep=';')
free  = pd.read_csv('../data/raw/free_text.csv',  sep=';')
freq  = pd.read_csv('../data/raw/frequency.csv',  sep=';')

print(f'Participants : {participants.shape}')
print(f'Fixed text   : {fixed.shape}')
print(f'Free text    : {free.shape}')
print(f'Frequency    : {freq.shape}')

## Inspect data types and missing values

In [ ]:
for name, df in [('Participants', participants), ('Fixed', fixed), ('Free', free), ('Frequency', freq)]:
    missing_pct = df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100
    print(f'{name:15s} | shape={df.shape} | missing={missing_pct:.1f}%')

## Coerce numeric columns

In [ ]:
num_cols = ['D1U1','D1D2','D1D3','D1U2','D1U3','U1D2','U1U2']
for c in num_cols:
    fixed[c] = pd.to_numeric(fixed[c], errors='coerce')
    if c in free.columns:
        free[c] = pd.to_numeric(free[c], errors='coerce')
freq['TotTime'] = pd.to_numeric(freq['TotTime'], errors='coerce')
print('Numeric coercion done')

## Remove invalid rows (negative/zero dwell or flight times)

In [ ]:
fixed_clean = fixed[fixed['D1U1'] > 0].copy()
fixed_clean = fixed_clean[fixed_clean['D1D2'] > 0]
fixed_clean = fixed_clean.dropna(subset=['D1U1','D1D2','emotionIndex'])
fixed_clean = fixed_clean.drop_duplicates()
print(f'Fixed after cleaning: {fixed_clean.shape}  (was {fixed.shape})')

## Save cleaned data

In [ ]:
fixed_clean.to_csv('../data/cleaned/cleaned_data.csv', index=False)
print('Saved: ../data/cleaned/cleaned_data.csv')